In [6]:
import requests
import random
import json
import time
from datetime import datetime

OUTPUT_SQL_FILE = "book_seed.sql"
TOTAL_RECORDS = 1000

SUBJECTS = [
    "fantasy",
    "science_fiction",
    "romance",
    "history",
    "mystery",
    "thriller",
    "horror",
    "biography",
    "technology",
    "programming",
    "philosophy",
    "psychology",
    "business",
    "economics",
    "politics",
    "self_help",
    "fiction",
    "young_adult",
    "children",
    "adventure"
]

LANGUAGES = ["en", "es", "fr", "de", "it", "ja"]
EDITIONS = [
    "1st Edition",
    "2nd Edition",
    "Revised Edition",
    "Collector's Edition",
    "Paperback",
    "Hardcover"
]

used_isbn13 = set()
used_isbn10 = set()


def sanitize(value: str) -> str:
    if value is None:
        return ""
    return value.replace("'", "''").replace("\x00", "")


def safe_json(data: dict) -> str:
    return sanitize(json.dumps(data, ensure_ascii=False))


def random_price():
    return round(random.uniform(5.99, 49.99), 2)


GENRES = [
    "Fantasy",
    "Sci-Fi",
    "Romance",
    "Mystery",
    "Thriller",
    "Programming",
    "Business",
    "Biography",
    "Psychology",
    "History",
    "Adventure",
    "Horror",
    "Young Adult"
]


def build_metadata(subject: str):
    return {
        "genre": random.choice(GENRES),
        "subject": subject,
        "price": random_price(),
        "rating": round(random.uniform(3.2, 5.0), 1),
        "in_stock": random.choice([True, False]),
        "format": random.choice(["Paperback", "Hardcover", "eBook"]),
        "tags": random.sample(
            [
                "popular",
                "classic",
                "award-winning",
                "bestseller",
                "recommended",
                "editor-pick",
                "fiction",
                "non-fiction"
            ],
            k=3
        )
    }


insert_statements = []


for subject in SUBJECTS:
    print(f"Fetching subject: {subject}")

    for page in range(1, 6):
        url = f"https://openlibrary.org/subjects/{subject}.json?limit=50&offset={(page - 1) * 50}"

        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"Failed fetching {subject}: {e}")
            continue

        works = data.get("works", [])

        for work in works:
            if len(insert_statements) >= TOTAL_RECORDS:
                break

            title = sanitize(work.get("title", "Unknown Title"))

            subtitle = None
            if random.random() > 0.6:
                subtitle = random.choice([
                    "A Novel",
                    "The Complete Guide",
                    "Essential Collection",
                    "An Epic Journey",
                    "Modern Edition"
                ])

            authors_raw = work.get("authors", [])
            authors = []

            for a in authors_raw:
                name = a.get("name")
                if name:
                    authors.append(sanitize(name))

            if not authors:
                authors = ["Unknown Author"]

            isbn13 = None
            isbn10 = None

            availability = work.get("availability", {})
            identifiers = work.get("cover_edition_key")

            generated13 = f"978{random.randint(1000000000, 9999999999)}"
            generated10 = str(random.randint(1000000000, 9999999999))

            while generated13 in used_isbn13:
                generated13 = f"978{random.randint(1000000000, 9999999999)}"

            while generated10 in used_isbn10:
                generated10 = str(random.randint(1000000000, 9999999999))

            used_isbn13.add(generated13)
            used_isbn10.add(generated10)

            isbn13 = generated13[:13]
            isbn10 = generated10[:10]

            publisher = sanitize(random.choice([
                "Penguin Random House",
                "HarperCollins",
                "Simon & Schuster",
                "O'Reilly Media",
                "Macmillan",
                "Oxford Press",
                "Scholastic",
                "Vintage Books",
                "No Starch Press"
            ]))

            description = sanitize(
                f"{title} is a compelling book in the {subject.replace('_', ' ')} category featuring engaging storytelling, rich concepts, and memorable characters."
            )

            language_code = random.choice(LANGUAGES)

            page_count = random.randint(80, 1200)

            year = random.randint(1950, 2025)
            month = random.randint(1, 12)
            day = random.randint(1, 28)

            publication_date = f"{year}-{month:02d}-{day:02d}"

            edition = random.choice(EDITIONS)

            metadata = build_metadata(subject)

            cover_id = work.get("cover_id")

            cover_image_url = None
            if cover_id:
                cover_image_url = f"https://covers.openlibrary.org/b/id/{cover_id}-L.jpg"
            else:
                cover_image_url = f"https://picsum.photos/seed/{isbn13}/400/600"

            authors_sql = ", ".join([f"'{a}'" for a in authors])

            subtitle_sql = f"'{sanitize(subtitle)}'" if subtitle else "NULL"

            sql = f"""
INSERT INTO book (
    isbn_13,
    isbn_10,
    title,
    subtitle,
    authors,
    publisher,
    description,
    language_code,
    page_count,
    publication_date,
    edition,
    metadata,
    cover_image_url
)
VALUES (
    '{isbn13}',
    '{isbn10}',
    '{title}',
    {subtitle_sql},
    ARRAY[{authors_sql}],
    '{publisher}',
    '{description}',
    '{language_code}',
    {page_count},
    '{publication_date}',
    '{sanitize(edition)}',
    '{safe_json(metadata)}'::jsonb,
    '{sanitize(cover_image_url)}'
);
""".strip()

            insert_statements.append(sql)

        if len(insert_statements) >= TOTAL_RECORDS:
            break

        time.sleep(0.5)

    if len(insert_statements) >= TOTAL_RECORDS:
        break


with open(OUTPUT_SQL_FILE, "w", encoding="utf-8") as f:
    f.write("-- Auto-generated PostgreSQL seed data\n")
    f.write(f"-- Generated at: {datetime.utcnow().isoformat()}Z\n\n")

    for statement in insert_statements:
        f.write(statement)
        f.write("\n\n")


print(f"Generated {len(insert_statements)} INSERT statements")
print(f"Output file: {OUTPUT_SQL_FILE}")


Fetching subject: fantasy
Fetching subject: science_fiction
Fetching subject: romance
Fetching subject: history
Generated 1000 INSERT statements
Output file: book_seed.sql


/tmp/ipykernel_5598/3603301778.py:257: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f.write(f"-- Generated at: {datetime.utcnow().isoformat()}Z\n\n")
